<a href="https://colab.research.google.com/github/wangyi-123456/114-2-/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [1]:
!pip install -q google-generativeai

In [2]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [17]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json
import tenacity # Import tenacity for retry mechanism

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [19]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1pwJRo3p-PrmV8SLAcoY2S2I6q5vGLtASjjh_262bVww/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [20]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.953317511s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

### 定義 AI 摘要函式

In [18]:
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
import google.genai.errors # Ensure google.genai.errors is available

@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10), # Exponential backoff: 4, 8, 10, 10, 10 seconds
    stop=stop_after_attempt(5), # Stop after 5 attempts
    reraise=True, # Re-raise the last exception if all retries fail
    retry=retry_if_exception_type(google.genai.errors.ServerError) # Retry only on ServerError (includes 503)
)
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [25]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    sheet_message = ""
    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    summary = "AI 摘要生成失敗。" # Default value for summary
    current_ai_sheet_write_status = "" # Status for writing AI summary to sheet

    try:
        # 獲取 AI 摘要
        generated_summary = get_ai_summary(new_grades_for_sheet)
        summary = generated_summary # If successful, update summary

        # 嘗試將 AI 摘要寫入 Google Sheet
        try:
            # 尋找第一行空白列來寫入 AI 摘要
            next_row = len(_ws.col_values(1)) + 1

            # 準備 AI 摘要要寫入的資料
            summary_lines = generated_summary.split('\n')
            summary_data_to_write = []

            # 第一行包含日期和「AI 摘要」標籤
            first_line_data = [datetime.now().strftime('%Y-%m-%d'), 'AI 摘要']
            if summary_lines:
                first_line_data.append(summary_lines[0])
            else:
                first_line_data.append('')
            summary_data_to_write.append(first_line_data)

            # 續後的行只包含摘要內容
            for line in summary_lines[1:]:
                summary_data_to_write.append(['', '', line])

            # 如果 summary_lines 為空，則只寫入日期和「AI 摘要」
            if not summary_lines:
                summary_data_to_write = [[datetime.now().strftime('%Y-%m-%d'), 'AI 摘要', '']]

            # 計算要更新的範圍
            num_rows_to_write = len(summary_data_to_write)
            if num_rows_to_write == 0: # Should not happen if summary is generated, but as a safeguard
                num_rows_to_write = 1

            # 確定寫入的範圍 (例如: A{next_row}:C{next_row + num_rows_to_write - 1})
            range_to_update = f"A{next_row}:C{next_row + num_rows_to_write - 1}"

            # 使用 update() 一次性寫入所有資料
            _ws.update(range_to_update, summary_data_to_write)
            current_ai_sheet_write_status = "AI 摘要已成功寫入 Google Sheet。"
        except Exception as e:
            current_ai_sheet_write_status = f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
            print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    except google.genai.errors.ServerError as e:
        summary = f"AI 模型呼叫失敗 (ServerError): {e}"
        print(summary)
        current_ai_sheet_write_status = "AI 摘要未能寫入 Google Sheet。" # Update status for sheet write
    except Exception as e: # Catch any other unexpected errors during AI summary generation
        summary = f"AI 模型呼叫失敗 (未知錯誤): {e}"
        print(summary)
        current_ai_sheet_write_status = "AI 摘要未能寫入 Google Sheet。" # Update status for sheet write

    # Append AI sheet write status to the overall sheet_message
    sheet_message += current_ai_sheet_write_status

    return sheet_message, summary

In [26]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")


--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---
呼叫 AI 時發生錯誤：429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 4.889023683s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

/tmp/ipykernel_619/1491915621.py:79: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  _ws.update(range_to_update, summary_data_to_write)


定義 Gradio 處理函式

In [27]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的科目和成績，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Row():
        with gr.Column():
            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=[["", ""]],
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="輸入科目與成績 (點擊最後一行可新增)"
            )
            submit_button = gr.Button("送出")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    submit_button.click(
        process_grades_and_summary,
        inputs=grade_input,
        outputs=[sheet_output, summary_output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a08737d2d752422143.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
